# Phase 1: Data Audit & Contracts

## Objective
Perform comprehensive data quality assessment and establish validation contracts.

## Dataset Review
- **Source**: Kaggle Credit Card Fraud Detection
- **Expected Shape**: 284,807 transactions x 31 features
- **Target**: Class (0 = legitimate, 1 = fraud)
- **Expected Imbalance**: ~0.172% fraud rate

In [ ]:
# Import require libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pprint import pprint
import great_expectations as gx
from great_expectations.expectations import (
    ExpectColumnToExist,
    ExpectTableColumnCountToEqual,
    ExpectColumnValuesToNotBeNull,
    ExpectColumnValuesToBeInTypeList,
    ExpectColumnValuesToBeBetween,
    ExpectColumnValuesToBeInSet,
    ExpectColumnMeanToBeBetween,
    ExpectColumnStdevToBeBetween,
    ExpectTableRowCountToBeBetween,
    ExpectColumnProportionOfUniqueValuesToBeBetween,
    ExpectColumnUniqueValueCountToBeBetween
)
import json
from great_expectations.exceptions import DataContextError
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Import our utility functions
from utils import (
    load_fraud_data,
    quick_data_summary,
    plot_target_distribution
)

# Import variables from utils.py
from utils import (
    DATA_RAW,
    DATA_PROCESSED,
    CONFIGS_DIR,
)

# Import file names from utils.py
from utils import (
    TXT_VALIDATION_SUMMARY,
    JSON_DATA_DICTIONARY,
    TXT_PHASE_1_COMPLETION_REPORT
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print("=" * 60)
print("PHASE 1: DATA AUDIT & CONTRACTS")
print("=" * 60)

print(f"\n📁 Data directory: {DATA_RAW}")
print("✅ Environment ready\n")

In [ ]:
# Load the dataset
print("=" * 60)
print("STEP 1: LOADING DATASET")
print("=" * 60)

df = load_fraud_data()

print("\n📊 Initial Data Check:")
print(f"\t• Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\t• Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\t• Features: {df.shape[1] - 1} (excluding target)")
print("\t• Target: Class")


## Step 2: Schema Validation

**Expected Schema:**
- `Time`: Numeric (seconds elapsed)
- `V1-V28`: Numeric (PCA components, 28 features)
- `Amount`: Numeric (transaction amount)
- `Class`: Binary (0=legitimate, 1=fraud)

**Total**: 31 columns, all numeric

In [ ]:
# Schema validation
print("=" * 60)
print("STEP 2: SCHEMA VALIDATION")
print("=" * 60)

# Expected columns
expected_columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
expected_count = 31

print("\n📋 Column Validation:")
print(f"\t• Expected columns: {expected_count}")
print(f"\t• Actual columns: {len(df.columns)}")
print(f"\t• Match: {'✅ YES' if len(df.columns) == expected_count else '❌ NO'}")

# Check for missing or extra columns
missing_cols = set(expected_columns) - set(df.columns)
extra_cols = set(df.columns) - set(expected_columns)

if missing_cols:
    print(f"\n⚠️ Missing columns: {missing_cols}")
if extra_cols:
    print(f"\n⚠️ Extra columns: {extra_cols}")
if not missing_cols and not extra_cols:
    print("\n✅ All expected columns present")

# Display column names for verification
print(f"\n📝 Actual Columns: ({len(df.columns)}):")
print(f"\t{list(df.columns)[:5]} ... {list(df.columns)[-3:]}")

In [ ]:
# Data type validation
print("=" * 60)
print("DATA TYPE ANALYSIS")
print("=" * 60)

print("\n🔢 Data Types Summary:")
print(df.dtypes.value_counts())

print("\n📊 Detailed Data Types:")
print(f"\t• Time: {df['Time'].dtype}")
print(f"\t• V1-V28: {df[[f'V{i}' for i in range(1, 29)]].dtypes.unique()}")
print(f"\t• Amount: {df['Amount'].dtype}")
print(f"\t• Class: {df['Class'].dtype}")

# Check if all columns are numeric
all_numeric = df.select_dtypes(include=[np.number]).shape[1] == df.shape[1]
print(f"\n✅ All columns numeric: {'YES' if all_numeric else 'NO'}")

# Verify Class is binary (0 and 1 only)
unique_classes = df["Class"].unique()
print("\n🎯 Target Variable (Class):")
print(f"\t• Unique values: {sorted(unique_classes)}")
print(f"\t• Is binary: {'✅ YES' if len(unique_classes) == 2 else '❌ NO'}")
print(f"\t• Valid range [0, 1]: {'✅ YES' if set(unique_classes).issubset({0, 1}) else '❌ NO'}")

In [ ]:
# Display first few rows to visually inspect structure
print("\n" + "=" * 60)
print("SAMPLE DATA INSPECTION")
print("=" * 60)

print("\n📄 First 3 rows:")
print(df.head(3))

print("\n📄 Last 3 rows:")
print(df.tail(3))

print("\n📄 Random 3 rows:")
print(df.sample(3, random_state=42))

print("\n✅ Schema validation complete - All checks passed!")

## Step 3: Data Quality Assessment

**Quality Checks:**
1. Missing values analysis
2. Duplicate records detection
3. Statistical summary (5-number summary + mean/std)
4. Feature range validation
5. Cardinality check

In [ ]:
# Missing values analysis
print("=" * 60)
print("STEP 3: DATA QUALITY ASSESSMENT")
print("=" * 60)

print("\n🔍 MISSING VALUES ANALYSIS")
print("-" * 60)

# Check for missing values
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": missing_pct
})

missing_df = missing_df[missing_df["Missing_Count"] > 0].sort_values("Missing_Count", ascending=False)

if len(missing_df) > 0:
    print(f"\n⚠️ Found missing values in {len(missing_df)} columns:")
    print(missing_df)
else:
    print("\n✅ No missing values detected in any column")
    print(f"\tTotal cells: {df.shape[0] * df.shape[1]:,}")
    print("\tAll cells populated: 100%")

# Check for any infinite values
print("\n🔍 INFINITE VALUES CHECK")
print("-" * 60)

inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()
total_inf = inf_counts.sum()

if total_inf > 0:
    print(f"⚠️ Found {total_inf} infinite values:")
    print(inf_counts[inf_counts > 0])
else:
    print("✅ No infinite values detected")


In [ ]:
# Duplicate records analysis
print("\n" + "=" * 60)
print("DUPLICATE RECORDS ANALYSIS")
print("=" * 60)

# Check for complete duplicate rows
duplicate_rows = df.duplicated().sum()
duplicate_pct = (duplicate_rows / len(df)) * 100

print("\n📊 Complete Duplicate Rows:")
print(f"\t• Count: {duplicate_rows:,}")
print(f"\t• Percentage: {duplicate_pct:.4f}%")

if duplicate_rows > 0:
    print(f"\t⚠️ Warning: Found {duplicate_rows:,} duplicate transactions")
    print("\t  Consider: Investigation needed - are these legitimate repeated transactions?")
else:
    print("\t✅ No complete duplicate rows")

# Check for duplicate transactions based on Time + Amount (potential duplicates)
# This is important for fraud detection - same amount at same time could be suspicious
print("\n📊 Potential Duplicate Transactions (Time + Amount):")
time_amount_duplicates = df.duplicated(subset=['Time', 'Amount'], keep=False).sum()
print(f"\t• Records with same Time AND Amount: {time_amount_duplicates:,}")
print(f"\t• Percentage: {(time_amount_duplicates / len(df)) * 100:.2f}%")

if time_amount_duplicates > 0:
    print("\tℹ️ Note: These may be legitimate (e.g., multiple $10 transactions)")
    print("\t  Will investigate further in EDA phase")


In [ ]:
# Show all duplicated rows (including all occurrences)
duplicates_df = df[df.duplicated(keep=False)]
print(f"Found {len(duplicates_df)} duplicated rows")

# Display first few duplicates for inspection
print(duplicates_df.head(10))

In [ ]:
# Comprehensive statistical summary
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)

# Use our utility function for quick summary
quick_data_summary(df)

print("\n" + "=" * 60)
print("DETAILED STATISTICS (5-NUMBER SUMMARY)")
print("=" * 60)

# Get descriptive statistics
stats = df.describe()
print(stats)


In [ ]:
# Feature range analysis
print("\n" + "=" * 60)
print("FEATURE RANGE ANALYSIS")
print("=" * 60)

# Key features to examine
key_features = ["Time", "Amount", "Class"]
v_features = [f"V{i}" for i in range(1, 29)]

print("\n📊 Key Features:")
for col in key_features:
    print(f"\n {col}:")
    print(f"\tMin:      {df[col].min():>15,.2f}")
    print(f"\tMax:      {df[col].max():>15,.2f}")
    print(f"\tRange:    {df[col].max() - df[col].min():>15,.2f}")
    print(f"\tMean:     {df[col].mean():>15,.2f}")
    print(f"\tMedian:   {df[col].median():>15,.2f}")

# V features summary (PCA components)
print("\n📊 PCA Features (V1 - V28):")
v_df = df[v_features]
print(f"\tMin across all:       {v_df.min().min():>10.3f}")
print(f"\tMax across all:       {v_df.max().max():>10.3f}")
print(f"\tMean (should be ~0):  {v_df.mean().mean():>10.6f}")
print(f"\tStd (should be ~1):   {v_df.std().std():>10.3f}")

# Check if V features are standardized (expected for PCA)
print("\nℹ️ V features appear to be PCA-transformed:")
print(f"\t• Mean ~= 0: {'✅' if abs(v_df.mean().mean()) < 0.01 else '❌'}")
print(f"\t• Std ~= 1: {'✅' if 0.8 < v_df.std().mean() < 1.2 else '❌'}")


In [ ]:
# Final quality assessment summary
print("=" * 60)
print("DATA QUALITY ASSESSMENT SUMMARY")
print("=" * 60)

quality_checks = {
    "No missing values": missing_counts.sum() == 0,
    "No infinite values": total_inf == 0,
    "No complete duplicates": duplicate_rows == 0,
    "All columns numeric": df.select_dtypes(include=[np.number]).shape[1] == df.shape[1],
    "Binary target (0, 1)" : set(df["Class"].unique()).issubset({0, 1}),
    "V features standardized": abs(v_df.mean().mean()) < 0.1 and 0.8 < v_df.std().mean() < 1.2,
    "Expected row count (~284K)": 280000 < len(df) < 290000,
    "Expected column count (31)": len(df.columns) == 31,
}

print(f"\n✅ Quality Checks Passed: {sum(quality_checks.values())} / {len(quality_checks)}\n")

for check, passed in quality_checks.items():
    icon = "✅" if passed else "❌"
    print(f"\t{icon} {check}")

# Calculate overall quality score
quality_score = (sum(quality_checks.values()) / len(quality_checks)) * 100

print(f"\n📊 Overall Data Quality Score: {quality_score:.1f}%")

if quality_score == 100:
    print("\tExceellent! Dataset is high quality and ready for analysis")
elif quality_score >= 80:
    print("\tGood quality - minor issues to address")
else:
    print("\tQuality concerns detected - review required")

print("\n✅ Step 3 complete - Data quality validated!")


In [ ]:
# Detailed duplicate analysis
print("=" * 60)
print("STEP 4: DUPLICATE INVESTIGATION")
print("=" * 60)

# Find duplicate rows
duplicate_mask = df.duplicated(keep=False)
duplicate_records = df[duplicate_mask].copy()

print("\n📊 Duplicate Records Analysis:")
print("  • Total duplicate records: {len(duplicate_records):,}")
print("  • Unique transactions (group): {len(duplicate_records) // 2:,}")
print("  • Percentage of datasets: {len(duplicate_records)/ len(df) * 100:.2f}%")

# Check class distribution in duplicates
print("\n🎯 Class Distribution in Duplicates:")
dup_class_dist = duplicate_records["Class"].value_counts().sort_index()
for class_val, count in dup_class_dist.items():
    pct = (count/ len(duplicate_records)) * 100
    label = "Legitimate" if class_val == 0 else "Fraud"
    print(f"  Class {class_val} ({label}): {count:,} ({pct:.2f}%)")

# Compare to overall distribution
print("\n📊 Comparison to Overall Distribution:")
overall_fraud_pct = (df["Class"].sum() / len(df)) * 100
dup_fraud_pct = (duplicate_records["Class"].sum() / len(duplicate_records)) * 100

print("  • Fraud rate in full dataset: {overall_fraud_pct:.2f}%")
print("  • Fraud rate in duplicates: {dup_fraud_pct:.2f}%")

if dup_fraud_pct > overall_fraud_pct * 1.5:
    print("  ⚠️ Duplicates contain HIGHER fraud rate. Investage further")
elif dup_fraud_pct < overall_fraud_pct * 0.5:
    print(" ℹ️ Duplicates contain LOWER fraud rate. Mostly legitimate")
else:
    print("  ✅ Duplicates have similar fraud rate to overall data")

In [ ]:
# Examine sample duplicates
print("=" * 60)
print("SAMPLE DUPLICATE RECORDS")
print("=" * 60)

# Get first duplicate group
if len(duplicate_records) > 0:
    # Find first set of duplicates
    first_dup_group = df[df.duplicated(keep=False)].head(10)
    
    print("\n📋 Example Duplicate Group (Shoing first 10 records):")
    print(first_dup_group[["Time", "V1", "V2", "V3", "Amount", "Class"]])
    
    # Check if these are exact duplicates or just Time+Amount duplicates
    print("\n🔍 Duplicate Type Analysis:")
    print("  • Complete duplicate rows detected: All 31 features are identical")
    print("  • Not just Time + Amount matches")
    
    # Decision guidance
    print("\n💡 Recommendation:")
    print("  • Complete duplicate rows detected")
    print("  • Could be: Data collection artrifacts, system errors, or legitimate repeated transactions")
    print("  • Strategy: Keep one copy of each duplicate, remove others")
    print("  • This preserves information while avoiding data leakage")
    print("  • Will remove duplicates in Phase 3: Data Cleaning")
else:
    print("  No duplicates to display")

In [ ]:
# Class imbalance deep dive with visualization
print("=" * 60)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 60)

# Calculate detailed imbalance metrics
class_counts = df["Class"].value_counts().sort_index()
total = len(df)

print("\n📊 Detailed Class Distribution:")
print("  Class 0 (Legitimate):")
print(f"  • Count: {class_counts[0]:,}")
print(f"  • Percentagee: {(class_counts[0]/total)*100:.4f}%")
print(f"\n  Class 1 (Fraud):")
print(f"  • Count: {class_counts[1]:,}")
print(f"  • Percentage: {(class_counts[1])*100:.4f}%")

# Imbalance ratio
imbalance_ratio = class_counts[1] / class_counts[0]
ratio_inverse = class_counts[0] / class_counts[1]

print("\nImbalance Metrics:")
print(f"  • Imbalance Ratio: {imbalance_ratio:.6f}%")
print(f"  • Ratio (Majority : Minority): {ratio_inverse:.1f}:1")
print(f"  • For every 1 fraud: {ratio_inverse:.0f} legitimate transactions")

# Severity assessment
print("\n🚨 Imbalance Severity: EXTREME")
print("  • Classification: Highly imbalanced (500:1)")
print("  • Impact: Standard classifiers will predict all as legitimate")
print("  • Required techniques:")
print("    - ✓ Use PR-AUC instead of accuracy")
print("    - ✓ Consider SMOTE/ADASYN oversampling")
print("    - ✓ Apply class weights in models")
print("    - ✓ Use stratifier sampling for validation")
print("    - ✓ Adjust decision threshold for business objectives")

# Use utility function to visualize
print("\n📊 Visualize class distribution...")
plot_target_distribution(df)

In [ ]:
# Calculate baseline metrics (what we need to beat)
print("=" * 60)
print("BASELINE METRICS (NAIVE APPROACH)")
print("=" * 60)

print("\n🎯 If we predict all transactions as legitimate (Class 0):")
print(f"  • Accuracy: {class_counts[0]/total*100:.2f}%")
print("  • Precision: Not defined (no positive predictions)")
print("  • Recall: 0% (catch no frauds)")
print("  • F1-Score: 0%")

print("\n💡 Key Insight:")
print(f"  • Accuracy is useless as a metric ({class_counts[0]/total*100:.2f}% by doing nothing)")
print("  • Must use PR-AUC, ROC-AUC, and Recall@Precision metrics")
print("  • Business goal: Maximize fraud detection while minimizing false positives")

print("\n💰 Business Cost Considerations:")
print("  • False Negative (miss fraud): High cost")
print("    - Direct financial loss")
print("    - Regulatory penalties")
print("    - Reputational damage")
print("\n  • False Positive (block legitimate): High cost")
print("    - Customer dissatisfaction")
print("    - Lost transactions")
print("    - Support overhead")
print("\n  ⚖️ Balance: Need high recall (catch frauds) at acceptable precision")


In [ ]:
# Summary of findings
print("=" * 60)
print("STEP 4 SUMMARY - KEY FINDINGS")
print("=" * 60)

findings = {
    "Duplicates": {
        "status": "⚠️ ATTENTION NEEDED",
        "details": [
            "1081 complete duplicate rows found (0.38%)",
            "Will remove in Phase 3: Data Cleaning",
            "Strategy: Keep first occurrence, drop duplicates"
        ]
    },
    "Class Imbalance": {
        "status": "🔴 EXTREME",
        "details": {
            "Fraud rate: 0.17% (577.9:1 ratio)",
            "Requires specialized techniques (SMOTE, class weights)",
            "PR-AUC is primary metric, not accuracy"
        }
    },
    "Data Quality": {
        "status": "✅ EXCELLENT",
        "details": [
            "Zero missing values",
            "No infinite values",
            "PCA features properly standardized",
            "All data types valid"
        ]
    },
    "Feature Engineering Needs": {
        "status": "ℹ️ IDENTIFIED",
        "details": {
            "Time feature spans 48 hours (2 days)",
            "Amount ranges: $0 - $25,691",
            "V1-V28 are PCA components (anonymous)",
            "May need temporal features in Phase 4"
        }
    }
}

for category, info in findings.items():
    print(f"\n{info['status']} {category}:")
    for detail in info["details"]:
        print(f"• {detail}")

print("\nStep 4 complete - Duplicates identified, imbalance quantified")
print("\nNext: Great Expecattions validation suite")

## Step 5: Great Expectations Validation Suite

**Validation Contracts:**
1. Schema validation (column names, types)
2. Completeness (no missing values)
3. Value ranges (min/max bounds)
4. Statistical properties (mean, std for PCA features)
5. Class distribution bounds
6. Uniqueness constraints

**Goal**: Create reusable validation suite for production monitoring

In [ ]:
# Create expectation suite
print("=" * 60)
print("STEP 5: CREATING EXPECTATION SUITE")
print("=" * 60)

suite_name = "fraud_detection_validation_suite"

# Create a new Expectation Suite
suite = gx.ExpectationSuite(name=suite_name)

print(f"📋 Created expectation suite: {suite_name}")

# Get pandas datasource
context = gx.get_context()

print(f"✅ Data context created: {context}")
print(f"✅ Ready to add expectations")


In [ ]:
# Define expectations: Schema and Completeness
print("=" * 60)
print("DEFINING EXPECTATIONS (1/4): SCHEMA & COMPLETENESS")
print("=" * 60)

expectations_list = []

# 1. Column count expectations
print("\n📋 Column Expectations:")
expectations_list.append(
    ExpectTableColumnCountToEqual(value=31)
)
print(f"  ✓ Table must have exactly 31 columns")

# 2. Expected columns present
expected_columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
for col in expected_columns:
    expectations_list.append(
        ExpectColumnToExist(column=col)
    )

print(f"  ✓ All {len(expected_columns)} expected columns must exist")

# 3. No missing values in any column
print("\n🔍 Completeness Expectations:")
for col in df.columns:
    expectations_list.append(
        ExpectColumnValuesToNotBeNull(column=col)
    )

print(f"  ✓ No null values allowed in any of {len(df.columns)} columns")

# 4. Data types
print("\n🔢 Data Type Expectations:")
for col in df.columns:
    if col == "Class":
        expectations_list.append(
            ExpectColumnValuesToBeInTypeList(
                column=col,
                type_list=["int64", "int32", "int"]
            )
        )
    else:
        expectations_list.append(
            ExpectColumnValuesToBeInTypeList(
                column=col,
                type_list=["float64", "float32", "float"]
            )
        )
        
print(f"  ✓ Data types validated for all columns")

print(f"\n📊 Expectations defined so far: {len(expectations_list)}")



In [ ]:
# Define expectations: Value ranges and constraints
print("=" * 60)
print("DEFINING EXPECTATIONS (2/4): VALUE RANGES")
print("=" * 60)

# Time constraints (0 to ~48 hours in seconds)
print("\n⏰ Time Constraints:")
expectations_list.append(
    ExpectColumnValuesToBeBetween(
        column="Time",
        min_value=0,
        max_value=200000
    )
)
print("  ✓ Time must be between 0 and 200,000 seconds")

# Amount constraints (non-negative, reasonable upper bound)
print("\n💰 Amount Constraints:")
expectations_list.append(
    ExpectColumnValuesToBeBetween(
        column="Amount",
        min_value=0,
        max_value=30000
    )
)
print("  ✓ Amount must be between $0 and $30,000")

# Class constraints (binary: 0 or 1)
print("\n🎯 Target Variable Constraints:")
expectations_list.append(
    ExpectColumnValuesToBeInSet(
        column="Class",
        value_set=[0, 1]
    )
)
expectations_list.append(
    ExpectColumnMeanToBeBetween(
        column="Class",
        min_value=0.0001,
        max_value=0.05
    )
)
print("  ✓ Class must be binary (0 or 1)")
print("  ✓ Fraud rate must be between 0.01% and 5%")

# V features constraints (standardized, should be mostly within -10 to 10)
print("\n📊 PCA Features Constraints:")
for i in range(1, 29):
    col = f"V{i}"
    expectations_list.append(
        ExpectColumnValuesToBeBetween(
            column=col,
            min_value=-150,
            max_value=150,
            mostly=0.999
        )
    )
print("  ✓ V1-V28 features constrained to reasonable ranges")

print(f"\n📊 Expectations defined so far: {len(expectations_list)}")


In [ ]:
# Define expectations - Statistical properties
print("=" * 60)
print("DEFINING EXPECTATIONS (3/4): STATISTICAL PROPERTIES")
print("=" * 60)

# PCA features should have mean ≈ 0 and std ≈ 1
print("\n📈 PCA Feature Statistics:")
for i in range(1, 29):
    col = f"V{i}"
    
    # Mean should be close to 0
    expectations_list.append(
        ExpectColumnMeanToBeBetween(
            column=col,
            min_value=-0.1,
            max_value=0.1
        )
    )
    
    # Std should be close to 1 (allow some variation)
    expectations_list.append(
        ExpectColumnStdevToBeBetween(
            column=col,
            min_value=0.5,
            max_value=2.0
        )
    )
print("  ✓ V1-V28 means constrained to [-0.1, 0.1]")
print("  ✓ V1-V28 standard deviations constrained to [0.5, 2.0]")

# Row count expectation (for production monitoring)
print("\n📏 Dataset Size:")
expectations_list.append(
    ExpectTableRowCountToBeBetween(
        min_value=100000,
        max_value=500000,
    )
)
print("  ✓ Dataset size must be between 100K and 500K rows")

print(f"📊 Expectations defined so far: {len(expectations_list)}")


In [ ]:
# Define expectations - Business logic
print("=" * 60)
print("DEFINING EXPECTATIONS (4/4): BUSINESS LOGIC")
print("=" * 60)

# Class imbalance check
print("\n⚖️ Class Balance:")
expectations_list.append(
    ExpectColumnProportionOfUniqueValuesToBeBetween(
        column="Class",
        min_value=0.00006,
        max_value=0.00007
    )
)
print("  ✓ Class column must have exactly 2 unique values (proper proportion)")

# Check for reasonable number of unique values in continuous features
print("\n🔢 Feature Cardinality:")
expectations_list.append(
    ExpectColumnUniqueValueCountToBeBetween(
        column="Amount",
        min_value=1000,
        max_value=None
    )
)
print("  ✓ Amount must have at least 1,000 unique values")

# Time should be sequential/increasing (check uniqueness)
expectations_list.append(
    ExpectColumnUniqueValueCountToBeBetween(
        column="Time",
        min_value=100,
        max_value=None
    )
)
print("  ✓ Time must have at least 100 unique values")

print(f"\n📊 Total expectations defined: {len(expectations_list)}")

# Add all expectations to suite
for expectation in expectations_list:
    suite.add_expectation(expectation)
    
print(f"\n✅ All {len(expectations_list)} expectations added to suite")

print("=" * 60)
print("📊 EXPECTATION SUITE COMPLETE")
print("=" * 60)
      

In [ ]:
# Validate the dataset against expectations
print("=" * 60)
print("RUNNING VALIDATION")
print("=" * 60)

print(f"Version: {gx.__version__}")
print("\n🔍 Running validation against current dataset...")

# Define or get pandas datasource
datasource_name = "fraud_dataset_pandas"
data_asset_name = "fraud_df_asset"
batch_definition_name = "fraud_df"

# Try to get existing, otherwise create new ones
try:
    data_source = context.data_sources.get(datasource_name)
except Exception:
    data_source = context.data_sources.add_pandas(name=datasource_name)

# Alwas recreate asset for fresh validation
if data_asset_name in [asset.name for asset in data_source.assets]:
    data_source.delete_asset(data_asset_name)

data_asset = data_source.add_dataframe_asset(name=data_asset_name)
batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)

# Get batch and validate
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
results = batch.validate(suite)

# Display results summary
print("=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

# Count successful and failed expectations
successful = sum(1 for result in results.results if result.success)
total = len(results.results)
failed = total - successful
success_rate = (successful / total * 100) if total > 0 else 0

print("\n📊 Validation Summary:")
print(f"  • Total Expectations: {total}")
print(f"  • Successful: {successful}")
print(f"  • Failed: {failed}")
print(f"  • Success Rate: {success_rate:.2f}%")

if results.success:
    print(f"\n✅ ALL VALIDATIONS PASSED.")
    print("  Dataset meets all quality requirements")
else:
    print(f"\n⚠️ Some validation failed ({failed} failures):")
    failed_expectations = [
        result for result in results.results
        if not result.success
    ]
    
    for i, result in enumerate(failed_expectations[:10], 1):
        print(f"\n  {i}. {result.expectation_config.type}")
        print(f"    Column: {result.expectation_config.kwargs.get('column', 'N/A')}")
        if hasattr(result, "result") and result.result:
            print(f"    Observed: {result.result.get('observed_value', 'N/A')}")
            print(f"    Expected: {result.expectation_config.kwargs}")


In [ ]:
# Create validation summary document
print("=" * 60)
print("STEP 5 SUMMARY: VALIDATION SUITE CREATED")
print("=" * 60)

validation_summary = f"""
✅ GREAT EXPECTATIONS VALIDATION SUITE COMPLETE

📋 Suite Name: {suite_name}
📅 Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
📦 Great Expectations Version: {gx.__version__}

📊 Validation Coverage ({total} expectations):
  1. Schema Validation
    • 31 column existence checks
    • 31 data type validations

  2. Completeness Checks
    • 31 null value checks (100% completeness required)
    
  3. Value Range Constraints
    • Time: [0, 200,000] seconds
    • Amount: [$0, $30,000]
    • Class: Binary [0, 1]
    • V1-V28: [-150, 150] with 99.9% compliance

  4. Statistical Properties
    • V1-V28 means: [-0.1, 0.1] (standardized)
    • V1-V28 std dev: [0.5, 2.0] (unit variance)
    • Dataset size: [100K, 500K] rows

 5. Business Logic
   • Class has exactly 2 unique values
   • Fraud rate: [0.01%, 5%]
   • Sufficient cardinality in Amount and Time

🎯 Purpose:
  • Production data quality monitoring
  • Automated validation for new data batches
  • Early detection of data drift
  • Documentation of data assumptions

💡Usage:
  • Run validation on new data before modeling
  • Monitor for schema or distribution changes
  • Alert on validation failures in production
  • Track data quality metrics over time

✅ Current Dataset Validation:
  • Total Expectations: {total}
  • Successful: {successful}
  • Failed: {failed}
  • Success Rate: {success_rate:.2f}%
"""

print(validation_summary)

# Save summary to a file
summary_path = CONFIGS_DIR / TXT_VALIDATION_SUMMARY # DATA_PROCESSED.parent.parent / "configs" / "validation_summary.txt"
summary_path.parent.mkdir(exist_ok=True)
with open(summary_path, "w") as f:
    f.write(validation_summary)
    
print(f"\n💾 Validation summary saved to: {summary_path}")

# Save expectation suite as JSON for reuse
suite_path = CONFIGS_DIR / f"{suite_name}.json" # DATA_PROCESSED.parent.parent / "configs" / f"{suite_name}.json"
suite_dict = suite.to_json_dict()

with open(suite_path, "w") as f:
    json.dump(suite_dict, f, indent=2)

print(f"💾 Expectation suite saved to: {suite_path}")

print("\n✅ Step 5 Complete: Data validation suite established.")


In [ ]:
# Document validation findings
print("=" * 60)
print("VALIDATION FINDINGS ANALYSIS")
print("=" * 60)

print("\n📊 Failed Validation Analysis:\n")

print("1. V26, V27, V28 Standard Deviations (3 failures)")
print("  • Observed: 0.33, 0.40, 0.48 (below expected 0.5)")
print("  • Explanation: Later PCA components have lower variance")
print("  • This is an expected behavior, not data quality issue")
print("  • Action: Document as normal, adjust expectation for production")
print("  • Impact: None, these features are still valid for modeling")

print("\n2. Class Proportion Expectation (1 failure)")
print("  • Observed: 7.02e-06 (proportion of unique values)")
print("  • Explanation: Overly restrictive constraint specification")
print("  • Class has 2 unique values as expected (0 and 1)")
print("  • Action: Adjust expectation range for production monitoring")
print("  • Impact: None, target variable is correct")

print("\n" + "=" * 60)
print("VALIDATION CONCLUSION")
print("=" * 60)

print(f"""
✅ VALIDATION ASSESSMENT: PASS (97.85%)

🎯 Data Quality Status: Exceelent
  • 182/186 expectations passed
  • 4 failures are not data quality issues
  • All critical validations passed:
    ✓ Schema correct (31 columns)
    ✓ No missing values
    ✓ No infinite values
    ✓ Data types valid
    ✓ Value ranges appropriate
    ✓ Target vartiable correct (binary 0/1)
    ✓ PCA features standardized (V1-V25 validated)

📝 Recommended Action:
  1. Adjust V26-V28 std expectations to [0.3, 2.0] for production
  2. Adjust Class propertion expectation to [1e-06, 1e-04]
  3. Document PCA variance distribution in data dictionary
  4. Proceed to Phase 2: EDA with confidence

🚀 Ready for Phase 2: No data quality blockers identified
""")

print("✅ Validation analysis complete.")


## Step 6: Comprehensive Data Dictionary

**Goal**: Document every feature with statistical properties, business context, and technical notes.

**Structure:**
- Feature name and type
- Statistical summary (min, max, mean, std, percentiles)
- Business interpretation
- Data quality notes
- Modeling considerations

In [ ]:
# Create comprehensive data dictionary
print("=" * 60)
print("STEP 6: DATA DICTIONARY CREATION")
print("=" * 60)

print("\n🔧 Generating comprehensive feature documentation...")

# Calculate detailed statistics for all features
data_dict_stats = {}

for col in df.columns:
    stats = {
        "dtype": str(df[col].dtype),
        "count": int(df[col].count()),
        "missing": int(df[col].isnull().sum()),
        "missing_pct": float((df[col].isnull().sum() / len(df)) * 100),
        "unique_values": int(df[col].nunique()),
        "min": float(df[col].min()),
        "max": float(df[col].max()),
        "mean": float(df[col].mean()),
        "std": float(df[col].std()),
        "median": float(df[col].median()),
        "q25": float(df[col].quantile(0.25)),
        "q75": float(df[col].quantile(0.75)),
        "skew": float(df[col].skew()),
        "kurtosis": float(df[col].kurtosis())
    }
    data_dict_stats[col] = stats

print(f"✅ Statistical summaries calculated for {len(data_dict_stats)} features")

for key, value in data_dict_stats.items():
    if key in ["Time", "Amount", "Class"]:
        print(f"\n{key} Stats:")
        for k, v in value.items():
            print(f"  {k}: {v}")


In [ ]:
# Create structured data dictionary with business context
print("=" * 60)
print("BUILDING STRUCTURED DATA DICTIONARY")
print("=" * 60)

# Define feature metadata (business context and notes)
feature_metadata = {
    "Time": {
        "category": "Temporal",
        "description": "Number of seconds elapsed between this transaction and the first transaction in the dataset",
        "unit": "seconds",
        "business_meaning": "Captures temporal patterns, fraud may occur at specific times of day or show temporal clustering",
        "data_quality": "Complete, no missing values. Spans 172,792 seconds (~48 hours)",
        "modeling_notes": "Consider engineering: hour of day, time since last transaction, transaction velocity"
    },
    "Amount": {
        "category": "Transaction",
        "description": "Transaction amount",
        "unit": "Currentcy",
        "business_meaning": "Transaction value, fraud patterns may differ by amount (e.g., small test charges vs large frauds)",
        "data_quality": "Complete, no missing values. Highly right-skewed (median $22, max $25,691)",
        "modeling_notes": "Consider log transformation, binning or normalization. Not standardized like V features"
    },
    "Class": {
        "category": "Target",
        "description": "Binary target variable indicating fraud",
        "unit": "categorical (0=legitimate, 1=fraud)",
        "business_meaning": "Ground truth labels: 0 for legitimate transactions, 1 for confirmed fraud",
        "data_quality": "Complete, Extreme imbalance: 0.17% fraud rate (577.9:1 ratio)",
        "modeling_notes": "Primary target. Requires class balancing techniques (SMOTE, class weights, thresholding tuning)"
    },
}

# Add V1-V28 metadata (PCA features)
for i in range(1, 29):
    col = f"V{i}"
    feature_metadata[col] = {
        "category": "PCA Component",
        "description": f"Principal Component {i}: PCA-transformed feature for confidentiality",
        "unit": "standardized (dimensionless)",
        "business_meaning": "Anonymized feature from PCA. Original features unknown due to privacy. Captures latent patterns in transaction data",
        "data_quality": f"Complete, standardized. Mean≈0, Std≈{data_dict_stats[col]['std']:.2f}",
        "modeling_notes": "Already standardized. No additional scaling needed. Lower-numbered components typically have more variance"
    }
    
print(f"✅ Business context defined for {len(feature_metadata)} features")


In [ ]:
# Generate formatted data dictionary
print("=" * 60)
print("DATA DICTIONARY: DETAILED FEATURE DOCUMENTATION")
print("=" * 60)

# Group features by category
categories = {
    "Temporal": ["Time"],
    "Transaction": ["Amount"],
    "Target": ["Class"],
    "PCA Component": [f"V{i}" for i in range(1, 29)]
}

for category, features in categories.items():
    print(f"\n{'=' * 60}")
    print(f"CATEGORY: {category.upper()}")
    print(f"{'=' * 60}")
    
    for col in features:
        if col not in feature_metadata:
            continue
        
        meta = feature_metadata[col]
        stats = data_dict_stats[col]
        
        print(f"\n📊 {col}")
        print(f"  {'-' * 55}")
        print(f"  {'Description:':<15} {meta['description']}")
        print(f"  {'Unit:':<15} {meta['unit']}")
        print(f"  {'Business Meaning:':<15} {meta['business_meaning'][:70]}...")
        print(f"\n  Statistical Summary:")
        print(f"  {'• Data Type:':<15} {stats['dtype']}")
        print(f"  {'• Count:':<15} {stats['count']:,}")
        print(f"  {'• Missing:':<15} {stats['missing']} ({stats['missing_pct']:.2f}%)")
        print(f"  {'• Unique Values:':<15} {stats['unique_values']:,}")
        print(f"  {'• Range:':<15} [{stats['min']:.3f}, {stats['max']:.3f}]")
        print(f"  {'• Mean:':<15} {stats['mean']:.3f}")
        print(f"  {'• Std Dev:':<15} {stats['std']:.3f}")
        print(f"  {'• Median:':<15} {stats['median']:.3f}")
        print(f"  {'• Q25-Q75:':<15} {stats['q25']:.3f}, {stats['q75']:.3f}")
        print(f"  {'• Skewness:':<15} {stats['skew']:.3f}")
        print(f"  {'• Kurtosis:':<15} {stats['kurtosis']:.3f}")
        print(f"\n  {'Data Quality:':<15} {meta['data_quality']}")
        print(f"  {'Modeling Notes:':<15} {meta['modeling_notes'][:70]}...")
        
        # Only show first 3 V features in detail, then summarize
        if category == "PCA Component" and col == "V3":
            print(f"\n    ... [V4-V28 follow same pattern - PCA components] ...")
            break

print(f"\n{'=' * 60}")


In [ ]:
# Create summary statistics table
print("=" * 60)
print("FEATURE STATISTICS SUMMARY TABLE")
print("=" * 60)

# Create a summary DataFrame
summary_data = []
for col in df.columns:
    stats = data_dict_stats[col]
    summary_data.append({
        "Feature": col,
        "Type": stats["dtype"],
        "Missing": stats["missing"],
        "Unique": stats["unique_values"],
        "Min": f"{stats['min']:.2f}",
        "Max": f"{stats['max']:.2f}",
        "Mean": f"{stats['mean']:.3f}",
        "Std": f"{stats['std']:.3f}",
        "skew": f"{stats['skew']:.2f}"
    })

summary_df = pd.DataFrame(summary_data)

print("\n📊 Complete Feature Statistics:\n")
print(summary_df.to_string(index=False))

print(f"\n✅ Summary table created for {len(summary_df)} features")


In [ ]:
# Identify features by variance (importance indicator for PCA)
print("=" * 60)
print("PCA COMPONENT VARIANCE ANALYSIS")
print("=" * 60)

# Calculate variance for V features
v_features = [f"V{i}" for i in range(1, 29)]
v_variances = [(col, data_dict_stats[col]["std"]**2) for col in v_features]
v_variances_sorted = sorted(v_variances, key=lambda x: x[1], reverse=True)

print("\n📊 Top 10 PCA Components by Variance (Likely Most Important):\n")
for i, (col, var) in enumerate(v_variances_sorted[:10], 1):
    print(f"  {i:2d}. {col}: Variance = {var:.4f}")

print(f"\n📊 Bottom 5 PCA Components by variance (Least Important):\n")
for i, (col, var) in enumerate(v_variances_sorted[-5:], 1):
    print(f"  {i}. {col}: Variance: {var:.4f}")
    
print(f"\n💡 Insight: Higher variance components (V1-V10) likely capture most signal")
print(f"  Lower variance components (V24-V28) may be less critical but still useful")


In [ ]:
# Create data dictionary (JSON format)
print("=" * 60)
print("EXPORTING DATA DICTIONARY")
print("=" * 60)

# Combine stats and metadata
full_data_dict = {
    "metadata": {
        "dataset_name": "Creadit Card Fraud Detection",
        "source": "Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud",
        "created_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_records": len(df),
        "total_features": len(df.columns),
        "time_priod": "2 days (September 2013)",
        "fraud_rate": f"{(df['Class'].sum() / len(df)*100):.4f}%",
        "imbalance_ratio": f"{(df['Class'].value_counts()[0] / df['Class'].value_counts()[1]):.1f}:1"
    },
    "features": {}
}

# Add each feature
for col in df.columns:
    full_data_dict["features"][col] = {
        "statistics": data_dict_stats[col],
        "metadata": feature_metadata.get(col, {})
    }

# Save to JSON
dict_path = CONFIGS_DIR / JSON_DATA_DICTIONARY # DATA_PROCESSED.parent.parent / "configs" / "data_dictionary.json"
with open(dict_path, "w") as f:
    json.dump(full_data_dict, f, indent=2)

print(f"💾 Data dictionary saved: {dict_path}")


In [ ]:
# Phase 1 completion summary
print("=" * 80)
print("PHASE 1: DATA AUDIT & CONTRACTS COMPLETE")
print("=" * 80)

completion_summary = f"""

✅ ALL PHASE 1 OBJECTIVES ACHIEVED

📋 Completed Activities:
  1. ✅ Dataset Loading & Validation
     • 284,807 transactions loaded successfully
     • 31 features validated (Time, V1-V28, Amount, Class)
  2. ✅ Schema Validation:
     • All expected columns present
     • Data types verified (30 float64, 1 int64)
 
  3. ✅ Data Quality Assessment
     • Zero missing values
     • No infinite values
     • 1,081 duplicate records identified (0.38%)
     • Quality score: 87.5% (Good)
  
  4. ✅ Class Imbalance Analysis
     • Fraud rate: 0.173% (577.9:1 ratio)
     • Extreme imbalance confirmed
     • Duplicates contain 10x higher fraud rate (1.73%)
  
  5. ✅ Great Expectations Validation Suite
     • 186 automated validation checks created
     • 182 validations passed (97.85% success rate)
     • 4 minor failures documented (not quality issues)
     • Production monitoring suite established
  
  6. ✅ Data Dictionary Created
     • Comprehensive documentation for 31 features
     • Statistical summaries calculated
     • Business context documented
     • Modeling notes included
     • Exported in JSON format
  
📊 Key Findings:
  • Dataset Quality: Excellent (no major issues)
  • Class Imbalance: Extreme (requires specialized techniques)
  • Duplicates: Present but manageable (will clean in Phase 3)
  • PCA Features: Properly standardized (V1-V25 validated)
  • Ready for EDA: No blockers identified

📁 Artifacts Created:
  • configs/validation_summary.txt
  • configs/fraud_detection_validation_suite.json
  • configs/data_dictionary.json

🎯 Success Criteria Met:
  ✅ Dataset loaded and validated
  ✅ No unexpected missing values or quality issues
  ✅ Great Expectations suite passed (97.85%)
  ✅ Comprehensive data dictionary created
  ✅ Documented assumptions and constraints
  ✅ Clear understanding of feature distributions

🔜 READY FOR PHASE 2: HIGH-IMPACT EDA
  • Target-feature relationships
  • DuckDB-.powered analysis
  • Temporal patterns investigation
  • Duplicate deep-dive
  • Outlier detection
  • Correlation analysis

📅 Phase 1 Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(completion_summary)

# Save Phase 1 completion report
report_path = CONFIGS_DIR / TXT_PHASE_1_COMPLETION_REPORT # DATA_PROCESSED.parent.parent / "configs" / "phase_1_completion_report.txt"
with open(report_path, "w") as f:
    f.write(completion_summary)

print(f"\n💾 Phase 1 completion report saved: {report_path}")
print(f"\n{'=' * 80}")
print("✅ Proceed to Phase 2: High-Impact EDA")
print(f"{'=' * 80}")
